# Matched-active MoE scaling with sorted top-2 dispatch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcoharuni/jax-addition-transformer/blob/main/notebooks/04_moe_scaling_t4.ipynb)

Four matched-active MoE transformers are trained from scratch at the same four horizons as notebook 02. Routing, checkpointing, evaluation, and dense comparison are visible below.


## Setup


In [ ]:
%pip install -q -U "jax[cuda12]==0.7.2" "flax==0.12.0" "optax==0.2.6" "numpy==2.3.3" "pandas==2.3.3" "matplotlib==3.10.7" "scipy==1.16.3"


In [ ]:
import gc
import json
import math
import os
import pickle
import shlex
import subprocess
import sys
import time
from pathlib import Path

CACHE_DIR = Path("/content/jax_compilation_cache/scaling_study")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
os.environ["JAX_COMPILATION_CACHE_DIR"] = str(CACHE_DIR)
os.environ["JAX_PERSISTENT_CACHE_MIN_COMPILE_TIME_SECS"] = "0"
# Keep this explanatory notebook process off the T4; workers remove this override.
os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
import pandas as pd
from flax import nnx
from google.colab import drive
from IPython.display import display

drive.mount("/content/drive")
REPOSITORY = "https://github.com/marcoharuni/jax-addition-transformer.git"
BRANCH = "dense-moe-scaling"
REPO_DIR = Path("/content/jax-addition-transformer")
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch",
                    REPOSITORY, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "merge", "--ff-only", f"origin/{BRANCH}"],
                   cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e",
                str(REPO_DIR), "--no-deps"], check=True)
WORKER = REPO_DIR / "scripts" / "scaling_worker.py"
DRIVE_ROOT = Path("/content/drive/MyDrive/jax-addition-transformer")
STUDY_ROOT = DRIVE_ROOT / "scaling"
FIGURE_ROOT = STUDY_ROOT / "figures"
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.2,
})

print("JAX:", jax.__version__)
print("Flax:", __import__("flax").__version__)
print("Optax:", optax.__version__)
print("Notebook devices:", jax.devices(), "(CPU by design; workers use the T4)")

assert jax.__version__ == "0.7.2"
assert __import__("flax").__version__ == "0.12.0"
assert optax.__version__ == "0.2.6"
assert WORKER.is_file()

def atomic_write_bytes(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_bytes(payload)
    os.replace(temporary, path)

def atomic_write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2) + "\n")
    os.replace(temporary, path)

def save_figure(figure, name):
    path = FIGURE_ROOT / f"{name}.png"
    figure.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


## Shared vocabulary and protocol

This repeats notebook 02's fixed split, hybrid sampler, answer-only loss, batch sizes, optimizer, precision, and endpoint validation so the comparison is self-contained.


In [ ]:
SEED = 42
VOCAB_SIZE = 13
MAX_SEQUENCE_LENGTH = 16
MODEL_INPUT_LENGTH = 15
PROMPT_LENGTH = 12
ANSWER_DIGITS = 4

TRAIN_SIZE = 200_000
VALIDATION_SIZE = 20_000
TEST_SIZE = 780_000
BATCH_SIZE = 2048
EVAL_BATCH_SIZE = 4000

HORIZONS = (50, 125, 300, 750)
PEAK_LR = 1e-3
FINAL_LR = 1e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
PARAM_DTYPE = jnp.float32
COMPUTE_DTYPE = jnp.float16

TOKENS = "0123456789 +="
TOKEN_TO_ID = {token: index for index, token in enumerate(TOKENS)}
ID_TO_TOKEN = {index: token for token, index in TOKEN_TO_ID.items()}

def encode(text):
    assert all(character in TOKEN_TO_ID for character in text)
    return np.asarray([TOKEN_TO_ID[character] for character in text], dtype=np.int32)

def decode(token_ids):
    token_ids = np.asarray(token_ids)
    assert token_ids.ndim == 1
    assert np.all((0 <= token_ids) & (token_ids < VOCAB_SIZE))
    return "".join(ID_TO_TOKEN[int(token_id)] for token_id in token_ids)

def format_prompt(a, b):
    assert 0 <= a <= 999 and 0 <= b <= 999
    return f"{a:03d} + {b:03d} = "

def format_example(a, b):
    answer = f"{a + b:04d}"
    return format_prompt(a, b) + answer[::-1]

for a, b in [(0, 0), (7, 42), (99, 1), (123, 456), (999, 999)]:
    example = format_example(a, b)
    assert len(example) == MAX_SEQUENCE_LENGTH
    print(example)

assert len(TOKENS) == VOCAB_SIZE
assert TRAIN_SIZE + VALIDATION_SIZE + TEST_SIZE == 1_000_000


In [ ]:
MOE_CONFIGS = {
    "moe_active_0p16m": {"dense_reference": "dense_0p16m", "n_layers": 2,
        "d_model": 64, "n_heads": 1, "dense_d_ff": 496, "expert_d_ff": 248,
        "dense_parameters": 162_176, "stored_parameters": 289_664,
        "active_parameters": 162_688},
    "moe_active_0p64m": {"dense_reference": "dense_0p64m", "n_layers": 2,
        "d_model": 128, "n_heads": 2, "dense_d_ff": 992, "expert_d_ff": 496,
        "dense_parameters": 643_840, "stored_parameters": 1_152_768,
        "active_parameters": 644_864},
    "moe_active_2p16m": {"dense_reference": "dense_2p16m", "n_layers": 3,
        "d_model": 192, "n_heads": 3, "dense_d_ff": 1488, "expert_d_ff": 744,
        "dense_parameters": 2_164_608, "stored_parameters": 3_881_088,
        "active_parameters": 2_166_912},
    "moe_active_10m": {"dense_reference": "dense_10m", "n_layers": 5,
        "d_model": 320, "n_heads": 5, "dense_d_ff": 2480, "expert_d_ff": 1240,
        "dense_parameters": 10_000_000, "stored_parameters": 17_942_400,
        "active_parameters": 10_006_400},
}
N_EXPERTS = 4
TOP_K = 2
ROUTER_BALANCE_COEFFICIENT = 1e-2
ROUTER_Z_LOSS_COEFFICIENT = 1e-3

study = {
    "model_sizes_per_architecture": len(MOE_CONFIGS),
    "independent_horizons": list(HORIZONS),
    "runs_per_architecture": len(MOE_CONFIGS) * len(HORIZONS),
}
assert study["model_sizes_per_architecture"] == 4
assert study["independent_horizons"] == [50, 125, 300, 750]
assert study["runs_per_architecture"] == 16

display(pd.DataFrame([
    {"model": name, "dense reference": config["dense_reference"],
     "layers": config["n_layers"], "width": config["d_model"],
     "expert width": config["expert_d_ff"], "stored parameters": config["stored_parameters"],
     "active proxy": config["active_parameters"]}
    for name, config in MOE_CONFIGS.items()
]).style.format({"stored parameters": "{:,}", "active proxy": "{:,}"}))


## Addition dataset


In [ ]:
def pair_ids_to_operands(pair_ids):
    pair_ids = np.asarray(pair_ids, dtype=np.int32)
    return pair_ids // 1000, pair_ids % 1000

def operand_lengths(values):
    values = np.asarray(values)
    return np.where(values < 10, 1, np.where(values < 100, 2, 3)).astype(np.int8)

def carry_codes(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    units = ((a % 10) + (b % 10) >= 10).astype(np.int8)
    tens = (((a // 10) % 10) + ((b // 10) % 10) + units >= 10).astype(np.int8)
    hundreds = (((a // 100) % 10) + ((b // 100) % 10) + tens >= 10).astype(np.int8)
    return (4 * units + 2 * tens + hundreds).astype(np.int8)

def stratum_codes(a, b):
    return (
        ((operand_lengths(a) - 1) * 3 + (operand_lengths(b) - 1)) * 8
        + carry_codes(a, b)
    ).astype(np.int16)

def make_sequences(pair_ids):
    pair_ids = np.asarray(pair_ids, dtype=np.int32)
    a, b = pair_ids_to_operands(pair_ids)
    total = a + b
    sequences = np.empty((len(pair_ids), MAX_SEQUENCE_LENGTH), dtype=np.uint8)
    sequences[:, 0] = a // 100
    sequences[:, 1] = (a // 10) % 10
    sequences[:, 2] = a % 10
    sequences[:, 3:6] = (10, 11, 10)
    sequences[:, 6] = b // 100
    sequences[:, 7] = (b // 10) % 10
    sequences[:, 8] = b % 10
    sequences[:, 9:12] = (10, 12, 10)
    sequences[:, 12] = total % 10
    sequences[:, 13] = (total // 10) % 10
    sequences[:, 14] = (total // 100) % 10
    sequences[:, 15] = (total // 1000) % 10
    return sequences

def largest_remainder(counts, total, capacity):
    ideal = counts.astype(np.float64) * total / counts.sum()
    allocated = np.minimum(np.floor(ideal).astype(np.int64), capacity)
    order = np.lexsort((np.arange(len(counts)), -(ideal - allocated)))
    remaining = total - int(allocated.sum())
    while remaining:
        eligible = order[allocated[order] < capacity[order]]
        take = eligible[:remaining]
        allocated[take] += 1
        remaining -= len(take)
    return allocated

def build_split(seed=SEED):
    pair_ids = np.arange(1_000_000, dtype=np.int32)
    a, b = pair_ids_to_operands(pair_ids)
    codes = stratum_codes(a, b)
    unique_codes, counts = np.unique(codes, return_counts=True)
    train_counts = largest_remainder(counts, TRAIN_SIZE, counts)
    validation_counts = largest_remainder(counts, VALIDATION_SIZE, counts - train_counts)
    rng = np.random.default_rng(seed)
    train_parts, validation_parts, test_parts = [], [], []
    for code_value, train_count, validation_count in zip(
        unique_codes, train_counts, validation_counts, strict=True
    ):
        members = pair_ids[codes == code_value].copy()
        rng.shuffle(members)
        train_parts.append(members[:train_count])
        validation_parts.append(members[train_count:train_count + validation_count])
        test_parts.append(members[train_count + validation_count:])
    train_ids = np.concatenate(train_parts)
    validation_ids = np.concatenate(validation_parts)
    test_ids = np.concatenate(test_parts)
    rng.shuffle(train_ids)
    rng.shuffle(validation_ids)
    rng.shuffle(test_ids)
    assert (len(train_ids), len(validation_ids), len(test_ids)) == (
        TRAIN_SIZE, VALIDATION_SIZE, TEST_SIZE
    )
    assert len(np.unique(np.concatenate([train_ids, validation_ids, test_ids]))) == 1_000_000
    return train_ids, validation_ids, test_ids

train_ids, validation_ids, test_ids = build_split()
train_sequences = make_sequences(train_ids)
train_a, train_b = pair_ids_to_operands(train_ids)
train_strata = stratum_codes(train_a, train_b)

print(f"Train: {len(train_ids):,}")
print(f"Validation: {len(validation_ids):,}")
print(f"Test: {len(test_ids):,}")
print(f"Cached training data: {train_sequences.nbytes / 1e6:.1f} MB")


In [ ]:
class HybridBatcher:
    def __init__(self, sequences, strata, batch_size, seed):
        self.sequences = sequences
        self.batch_size = batch_size
        self.rng = np.random.default_rng(seed)
        self.stratum_indices = [
            np.flatnonzero(strata == code) for code in np.unique(strata)
        ]

    def sample(self):
        natural_count = self.batch_size // 2
        balanced_count = self.batch_size - natural_count
        natural = self.rng.integers(0, len(self.sequences), size=natural_count)
        chosen_strata = self.rng.integers(0, len(self.stratum_indices), size=balanced_count)
        balanced = np.empty(balanced_count, dtype=np.int64)
        for stratum in np.unique(chosen_strata):
            locations = np.flatnonzero(chosen_strata == stratum)
            balanced[locations] = self.rng.choice(
                self.stratum_indices[int(stratum)], size=len(locations), replace=True
            )
        indices = np.concatenate([natural, balanced])
        self.rng.shuffle(indices)
        batch = self.sequences[indices].astype(np.int32)
        return batch[:, :-1], batch[:, 1:]

sample_batcher = HybridBatcher(train_sequences, train_strata, BATCH_SIZE, SEED)
sample_inputs, sample_targets = sample_batcher.sample()
assert sample_inputs.shape == sample_targets.shape == (BATCH_SIZE, MODEL_INPUT_LENGTH)

figure, axes = plt.subplots(1, 2, figsize=(12, 3.5))
example = format_example(123, 456)
for position, token in enumerate(example):
    axes[0].add_patch(plt.Rectangle((position, 0), 0.9, 0.8, fill=False))
    axes[0].text(position + 0.45, 0.4, "space" if token == " " else token,
                 ha="center", va="center", fontsize=8)
axes[0].set(xlim=(0, 16), ylim=(0, 1), title="123 + 456 = 9750")
axes[0].axis("off")

sample_ids = train_ids[np.random.default_rng(SEED).integers(0, len(train_ids), 50_000)]
sample_a, sample_b = pair_ids_to_operands(sample_ids)
natural = np.bincount(carry_codes(sample_a, sample_b), minlength=8) / 50_000
batch_a = sample_inputs[:, 0] * 100 + sample_inputs[:, 1] * 10 + sample_inputs[:, 2]
batch_b = sample_inputs[:, 6] * 100 + sample_inputs[:, 7] * 10 + sample_inputs[:, 8]
hybrid = np.bincount(carry_codes(batch_a, batch_b), minlength=8) / BATCH_SIZE
x = np.arange(8)
axes[1].bar(x - 0.2, natural, 0.4, label="natural")
axes[1].bar(x + 0.2, hybrid, 0.4, label="training batch")
axes[1].set_xticks(x, [f"{value:03b}" for value in x])
axes[1].set(xlabel="carry pattern", ylabel="fraction", title="Carry coverage")
axes[1].legend()
figure.tight_layout()
save_figure(figure, "addition_data_protocol")


## Shared transformer primitives


In [ ]:
def normal_parameter(rngs, shape, scale):
    values = jax.random.normal(rngs.params(), shape, dtype=jnp.float32) * scale
    return nnx.Param(values.astype(PARAM_DTYPE))

def matrix_multiply(x, weight):
    x16 = x.astype(COMPUTE_DTYPE)
    w16 = weight.astype(COMPUTE_DTYPE)
    return jax.lax.dot_general(
        x16, w16, (((x16.ndim - 1,), (0,)), ((), ())),
        preferred_element_type=jnp.float32,
    )

class Linear(nnx.Module):
    def __init__(self, input_width, output_width, rngs, scale=0.02):
        self.kernel = normal_parameter(rngs, (input_width, output_width), scale)

    def __call__(self, x):
        return matrix_multiply(x, self.kernel.value)

class LayerNorm(nnx.Module):
    def __init__(self, width):
        self.scale = nnx.Param(jnp.ones((width,), dtype=PARAM_DTYPE))
        self.bias = nnx.Param(jnp.zeros((width,), dtype=PARAM_DTYPE))

    def __call__(self, x):
        x = x.astype(jnp.float32)
        mean = jnp.mean(x, axis=-1, keepdims=True)
        variance = jnp.mean(jnp.square(x - mean), axis=-1, keepdims=True)
        return (x - mean) * jax.lax.rsqrt(variance + 1e-5) * self.scale.value + self.bias.value

def gelu(x):
    return 0.5 * x * (1.0 + jax.lax.erf(x / math.sqrt(2.0)))

CAUSAL_MASK = jnp.tril(jnp.ones((MODEL_INPUT_LENGTH, MODEL_INPUT_LENGTH), dtype=bool))

class CausalMHA(nnx.Module):
    def __init__(self, config, rngs):
        width = config["d_model"]
        residual_scale = 0.02 / math.sqrt(2 * config["n_layers"])
        self.width = width
        self.heads = config["n_heads"]
        self.head_dim = width // self.heads
        self.q_proj = Linear(width, width, rngs)
        self.k_proj = Linear(width, width, rngs)
        self.v_proj = Linear(width, width, rngs)
        self.out_proj = Linear(width, width, rngs, scale=residual_scale)

    def __call__(self, x):
        batch, length, _ = x.shape
        q = self.q_proj(x).reshape(batch, length, self.heads, self.head_dim)
        k = self.k_proj(x).reshape(batch, length, self.heads, self.head_dim)
        v = self.v_proj(x).reshape(batch, length, self.heads, self.head_dim)
        scores = jnp.einsum(
            "bthd,bshd->bhts", q.astype(jnp.float32), k.astype(jnp.float32),
            preferred_element_type=jnp.float32,
        ) / math.sqrt(self.head_dim)
        scores = jnp.where(
            CAUSAL_MASK[None, None, :length, :length],
            scores,
            jnp.finfo(jnp.float32).min,
        )
        probabilities = jax.nn.softmax(scores, axis=-1)
        attended = jnp.einsum(
            "bhts,bshd->bthd", probabilities, v.astype(jnp.float32),
            preferred_element_type=jnp.float32,
        )
        return self.out_proj(attended.reshape(batch, length, self.width))

class ModuleSequence(nnx.Module):
    def __init__(self, layers):
        self.length = len(layers)
        for index, layer in enumerate(layers):
            setattr(self, f"layer_{index}", layer)

    def __iter__(self):
        return (getattr(self, f"layer_{index}") for index in range(self.length))


## Sorted top-2 MoE dispatch


In [ ]:
class RaggedTop2MoE(nnx.Module):
    def __init__(self, config, rngs):
        width = config["d_model"]
        expert_width = config["expert_d_ff"]
        residual_scale = 0.02 / math.sqrt(2 * config["n_layers"])
        self.width = width
        self.router = Linear(width, N_EXPERTS, rngs)
        self.up_kernel = normal_parameter(rngs, (N_EXPERTS, width, expert_width), 0.02)
        self.down_kernel = normal_parameter(
            rngs, (N_EXPERTS, expert_width, width), residual_scale
        )

    def __call__(self, x):
        original_shape = x.shape
        flat_x = x.reshape(-1, self.width).astype(jnp.float32)
        token_count = flat_x.shape[0]
        router_logits = self.router(flat_x).astype(jnp.float32)
        router_probabilities = jax.nn.softmax(router_logits, axis=-1)
        top_logits, top_expert_ids = jax.lax.top_k(router_logits, TOP_K)
        top_gate_weights = jax.nn.softmax(top_logits, axis=-1)

        route_token_ids = jnp.repeat(jnp.arange(token_count, dtype=jnp.int32), TOP_K)
        route_expert_ids = top_expert_ids.reshape(-1).astype(jnp.int32)
        route_weights = top_gate_weights.reshape(-1).astype(jnp.float32)
        route_order = jnp.argsort(route_expert_ids, stable=True)
        sorted_token_ids = route_token_ids[route_order]
        sorted_expert_ids = route_expert_ids[route_order]
        sorted_weights = route_weights[route_order]
        sorted_inputs = flat_x[sorted_token_ids].astype(COMPUTE_DTYPE)
        group_sizes = jnp.bincount(sorted_expert_ids, length=N_EXPERTS).astype(jnp.int32)

        hidden = jax.lax.ragged_dot(
            sorted_inputs, self.up_kernel.value.astype(COMPUTE_DTYPE), group_sizes,
            preferred_element_type=jnp.float32,
        )
        hidden = gelu(hidden)
        sorted_outputs = jax.lax.ragged_dot(
            hidden.astype(COMPUTE_DTYPE),
            self.down_kernel.value.astype(COMPUTE_DTYPE),
            group_sizes,
            preferred_element_type=jnp.float32,
        )
        combined = jnp.zeros((token_count, self.width), dtype=jnp.float32)
        combined = combined.at[sorted_token_ids].add(sorted_outputs * sorted_weights[:, None])

        top1_fraction = jnp.mean(
            jax.nn.one_hot(top_expert_ids[:, 0], N_EXPERTS, dtype=jnp.float32), axis=0
        )
        route_fraction = jnp.mean(
            jax.nn.one_hot(route_expert_ids, N_EXPERTS, dtype=jnp.float32), axis=0
        )
        router_importance = jnp.mean(router_probabilities, axis=0)
        balance_loss = N_EXPERTS * jnp.sum(top1_fraction * router_importance)
        z_loss = jnp.mean(jnp.square(jax.nn.logsumexp(router_logits, axis=-1)))
        entropy = -jnp.mean(jnp.sum(
            router_probabilities * jnp.log(jnp.maximum(router_probabilities, 1e-9)), axis=-1
        ))
        diagnostics = {
            "balance_loss": balance_loss, "z_loss": z_loss,
            "router_entropy": entropy, "top1_fraction": top1_fraction,
            "route_fraction": route_fraction, "router_importance": router_importance,
        }
        return combined.reshape(original_shape), diagnostics

class MoETransformerBlock(nnx.Module):
    def __init__(self, config, rngs):
        self.attention_norm = LayerNorm(config["d_model"])
        self.attention = CausalMHA(config, rngs)
        self.moe_norm = LayerNorm(config["d_model"])
        self.moe = RaggedTop2MoE(config, rngs)

    def __call__(self, x):
        x = x + self.attention(self.attention_norm(x))
        moe_output, diagnostics = self.moe(self.moe_norm(x))
        return x + moe_output, diagnostics

class MoEAdditionTransformer(nnx.Module):
    def __init__(self, config, rngs):
        width = config["d_model"]
        self.token_embedding = normal_parameter(rngs, (VOCAB_SIZE, width), 0.02)
        self.position_embedding = normal_parameter(rngs, (MODEL_INPUT_LENGTH, width), 0.02)
        self.blocks = ModuleSequence([
            MoETransformerBlock(config, rngs) for _ in range(config["n_layers"])
        ])
        self.final_norm = LayerNorm(width)

    def __call__(self, token_ids, return_routing=False):
        x = self.token_embedding.value[token_ids]
        x = x + self.position_embedding.value[None, :, :]
        layers = []
        for block in self.blocks:
            def apply_block(value, current_block=block):
                return current_block(value)
            x, diagnostics = jax.checkpoint(apply_block)(x)
            layers.append(diagnostics)
        x = self.final_norm(x)
        logits = jnp.einsum(
            "btd,vd->btv", x.astype(jnp.float32),
            self.token_embedding.value.astype(jnp.float32),
            preferred_element_type=jnp.float32,
        )
        if not return_routing:
            return logits
        routing = {key: jnp.stack([layer[key] for layer in layers]) for key in layers[0]}
        for key in ("balance_loss", "z_loss", "router_entropy"):
            routing[key] = jnp.mean(routing[key])
        return logits, routing

def moe_parameter_formulas(config):
    layers, width, expert_width = config["n_layers"], config["d_model"], config["expert_d_ff"]
    shared = layers * (4 * width * width + 4 * width) + (
        VOCAB_SIZE * width + MODEL_INPUT_LENGTH * width + 2 * width
    )
    router = layers * width * N_EXPERTS
    stored = shared + router + layers * N_EXPERTS * 2 * width * expert_width
    active = shared + router + layers * TOP_K * 2 * width * expert_width
    return stored, active

parameter_rows = []
for model_id, config in MOE_CONFIGS.items():
    model = MoEAdditionTransformer(config, nnx.Rngs(params=SEED))
    real_count = sum(int(leaf.size) for leaf in jax.tree.leaves(nnx.state(model, nnx.Param)))
    stored, active = moe_parameter_formulas(config)
    assert real_count == stored == config["stored_parameters"]
    assert active == config["active_parameters"]
    parameter_rows.append({"model": model_id, "tree count": real_count,
                           "stored formula": stored, "active proxy": active})
    del model
    gc.collect()

display(pd.DataFrame(parameter_rows).style.format({
    "tree count": "{:,}", "stored formula": "{:,}", "active proxy": "{:,}",
}))


## Loss, generation, training, and validation


In [ ]:
ANSWER_MASK = jnp.arange(MODEL_INPUT_LENGTH) >= MODEL_INPUT_LENGTH - ANSWER_DIGITS

def answer_loss(logits, targets):
    log_probabilities = jax.nn.log_softmax(logits.astype(jnp.float32), axis=-1)
    token_losses = -jnp.take_along_axis(
        log_probabilities, targets[..., None], axis=-1
    ).squeeze(-1)
    denominator = targets.shape[0] * ANSWER_DIGITS
    loss = jnp.sum(jnp.where(ANSWER_MASK[None, :], token_losses, 0.0)) / denominator
    predictions = jnp.argmax(logits, axis=-1)
    accuracy = jnp.sum(
        jnp.where(ANSWER_MASK[None, :], predictions == targets, False)
    ) / denominator
    return loss, accuracy

def greedy_generate(model, prompts):
    buffer = jnp.zeros((prompts.shape[0], MODEL_INPUT_LENGTH), dtype=jnp.int32)
    buffer = buffer.at[:, :PROMPT_LENGTH].set(prompts)

    def generate_digit(current_buffer, offset):
        logits = model(current_buffer)
        next_token = jnp.argmax(logits[:, PROMPT_LENGTH + offset - 1, :], axis=-1)
        current_buffer = jax.lax.cond(
            offset < ANSWER_DIGITS - 1,
            lambda value: value.at[:, PROMPT_LENGTH + offset].set(next_token),
            lambda value: value,
            current_buffer,
        )
        return current_buffer, next_token.astype(jnp.int32)

    _, generated = jax.lax.scan(generate_digit, buffer, jnp.arange(ANSWER_DIGITS))
    generated = jnp.swapaxes(generated, 0, 1)
    return generated, jnp.all(generated < 10, axis=-1)

def make_learning_rate(horizon):
    warmup = max(1, horizon // 10)
    schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0,
        peak_value=PEAK_LR,
        warmup_steps=warmup,
        decay_steps=horizon - 1,
        end_value=FINAL_LR,
    )
    assert float(schedule(0)) == 0.0
    np.testing.assert_allclose(float(schedule(horizon - 1)), FINAL_LR, rtol=1e-5)
    return schedule

def path_parts(path):
    return tuple(str(getattr(entry, "key", getattr(entry, "idx", entry))) for entry in path)

def all_finite(tree):
    return jnp.all(jnp.stack([jnp.all(jnp.isfinite(leaf)) for leaf in jax.tree.leaves(tree)]))


In [ ]:
MOE_ROOT = DRIVE_ROOT / "scaling" / "notebook_04_moe"
MOE_RUNS_ROOT = MOE_ROOT / "runs"
MOE_RESULTS_PATH = MOE_ROOT / "moe_results.json"
DENSE_RESULTS_PATH = DRIVE_ROOT / "scaling" / "notebook_02_dense" / "dense_results.json"
COMPARISON_ROOT = DRIVE_ROOT / "scaling" / "dense_moe_comparison"
MOE_RUNS_ROOT.mkdir(parents=True, exist_ok=True)
COMPARISON_ROOT.mkdir(parents=True, exist_ok=True)

def build_moe_training(config, horizon):
    model = MoEAdditionTransformer(config, nnx.Rngs(params=SEED))
    graphdef, params = nnx.split(model, nnx.Param)
    decay_mask = jax.tree_util.tree_map_with_path(
        lambda path, leaf: leaf.ndim >= 2 and (
            "attention" in path_parts(path)
            or "up_kernel" in path_parts(path)
            or "down_kernel" in path_parts(path)
        ),
        params,
    )
    learning_rate = make_learning_rate(horizon)
    optimizer = optax.chain(
        optax.clip_by_global_norm(GRAD_CLIP),
        optax.scale_by_adam(b1=0.9, b2=0.99, eps=1e-8),
        optax.masked(optax.add_decayed_weights(WEIGHT_DECAY), decay_mask),
        optax.scale_by_learning_rate(learning_rate),
    )
    optimizer_state = optimizer.init(params)

    @jax.jit
    def train_step(params, optimizer_state, inputs, targets):
        def objective(candidate_params):
            model = nnx.merge(graphdef, candidate_params)
            logits, routing = model(inputs, return_routing=True)
            language_loss, accuracy = answer_loss(logits, targets)
            total = (
                language_loss
                + ROUTER_BALANCE_COEFFICIENT * routing["balance_loss"]
                + ROUTER_Z_LOSS_COEFFICIENT * routing["z_loss"]
            )
            return total, {"language_loss": language_loss,
                           "answer_token_accuracy": accuracy, **routing}
        (loss, auxiliary), gradients = jax.value_and_grad(objective, has_aux=True)(params)
        gradient_norm = optax.global_norm(gradients)
        updates, new_optimizer_state = optimizer.update(gradients, optimizer_state, params)
        new_params = optax.apply_updates(params, updates)
        finite = jnp.isfinite(loss) & all_finite(gradients) & all_finite(new_params)
        safe_params = jax.tree.map(lambda new, old: jnp.where(finite, new, old),
                                   new_params, params)
        safe_state = jax.tree.map(lambda new, old: jnp.where(finite, new, old),
                                  new_optimizer_state, optimizer_state)
        return safe_params, safe_state, {
            "loss": loss, "gradient_norm": gradient_norm, "finite": finite, **auxiliary
        }

    @jax.jit
    def evaluate_batch(params, inputs, targets):
        model = nnx.merge(graphdef, params)
        logits, routing = model(inputs, return_routing=True)
        loss, accuracy = answer_loss(logits, targets)
        return loss, accuracy, routing

    @jax.jit
    def generate_batch(params, prompts):
        return greedy_generate(nnx.merge(graphdef, params), prompts)

    return params, optimizer_state, learning_rate, train_step, evaluate_batch, generate_batch

def evaluate_moe_validation(params, evaluate_batch, generate_batch):
    started = time.perf_counter()
    loss_total = token_total = exact_total = 0.0
    routing_batches = []
    for start in range(0, len(validation_ids), EVAL_BATCH_SIZE):
        ids = validation_ids[start:start + EVAL_BATCH_SIZE]
        sequences = make_sequences(ids).astype(np.int32)
        inputs = jnp.asarray(sequences[:, :-1])
        targets = jnp.asarray(sequences[:, 1:])
        loss, token_accuracy, routing = evaluate_batch(params, inputs, targets)
        generated, valid = generate_batch(params, inputs[:, :PROMPT_LENGTH])
        generated, valid = np.asarray(generated), np.asarray(valid)
        exact = valid & np.all(generated == sequences[:, -ANSWER_DIGITS:], axis=1)
        loss_total += float(loss) * len(ids)
        token_total += float(token_accuracy) * len(ids)
        exact_total += int(exact.sum())
        routing_batches.append(jax.device_get(routing))
    routing = {
        key: np.mean(np.stack([np.asarray(batch[key]) for batch in routing_batches]), axis=0)
        for key in routing_batches[0]
    }
    return {
        "validation_loss": loss_total / len(validation_ids),
        "validation_token_accuracy": token_total / len(validation_ids),
        "validation_exact_match": exact_total / len(validation_ids),
        "validation_seconds": time.perf_counter() - started,
        "router_entropy": float(routing["router_entropy"]),
        "balance_loss": float(routing["balance_loss"]),
        "z_loss": float(routing["z_loss"]),
        "top1_fraction": np.asarray(routing["top1_fraction"]).tolist(),
        "route_fraction": np.asarray(routing["route_fraction"]).tolist(),
        "router_importance": np.asarray(routing["router_importance"]).tolist(),
    }

def run_moe_experiment(model_id, config, horizon):
    run_id = f"{model_id}_h{horizon:04d}"
    run_config = {"run_id": run_id, "model_id": model_id, "horizon": horizon,
                  "seed": SEED, "batch_size": BATCH_SIZE, "model": config,
                  "experts": N_EXPERTS, "top_k": TOP_K}
    run_dir = MOE_RUNS_ROOT / run_id
    latest_path = run_dir / "latest_checkpoint.pkl"
    best_path = run_dir / "best_checkpoint.pkl"
    history_path = run_dir / "history.json"
    result_path = run_dir / "result.json"
    run_dir.mkdir(parents=True, exist_ok=True)

    if result_path.exists():
        completed = json.loads(result_path.read_text())
        if completed.get("status") == "complete" and completed.get("run_config") == run_config:
            print(f"skip {run_id}: complete")
            return completed
        raise ValueError(f"existing result does not match this run: {run_id}")

    params, optimizer_state, learning_rate, train_step, evaluate_batch, generate_batch = (
        build_moe_training(config, horizon)
    )
    batcher = HybridBatcher(train_sequences, train_strata, BATCH_SIZE, SEED)
    history, start_step = [], 0
    optimizer_seconds = compilation_seconds = 0.0
    if latest_path.exists():
        checkpoint = pickle.loads(latest_path.read_bytes())
        if checkpoint["run_config"] != run_config:
            raise ValueError(f"checkpoint configuration mismatch: {run_id}")
        params = jax.tree.map(jnp.asarray, checkpoint["params"])
        optimizer_state = jax.tree.map(jnp.asarray, checkpoint["optimizer_state"])
        batcher.rng.bit_generator.state = checkpoint["batcher_rng_state"]
        history, start_step = checkpoint["history"], checkpoint["step"]
        optimizer_seconds = checkpoint["optimizer_seconds"]
        compilation_seconds = checkpoint["compilation_seconds"]
        print(f"resume {run_id} after step {start_step}")
    else:
        print(f"start {run_id}")

    checkpoint_every, log_every = min(50, horizon), min(25, horizon)
    for step in range(start_step + 1, horizon + 1):
        inputs, targets = batcher.sample()
        before = time.perf_counter()
        params, optimizer_state, metrics = train_step(
            params, optimizer_state, jnp.asarray(inputs), jnp.asarray(targets)
        )
        jax.block_until_ready(metrics["loss"])
        elapsed = time.perf_counter() - before
        optimizer_seconds += elapsed
        if step == start_step + 1:
            compilation_seconds += elapsed
        if step == start_step + 1 or step % log_every == 0 or step == horizon:
            host = jax.device_get(metrics)
            assert bool(host["finite"]), f"non-finite values at {run_id} step {step}"
            record = {
                "step": step, "loss": float(host["loss"]),
                "language_loss": float(host["language_loss"]),
                "answer_token_accuracy": float(host["answer_token_accuracy"]),
                "gradient_norm": float(host["gradient_norm"]),
                "balance_loss": float(host["balance_loss"]),
                "z_loss": float(host["z_loss"]),
                "router_entropy": float(host["router_entropy"]),
                "route_fraction": np.asarray(host["route_fraction"]).tolist(),
                "learning_rate": float(learning_rate(step - 1)),
                "step_seconds": elapsed,
            }
            history.append(record)
            print(
                f"{run_id} | step {step:4d}/{horizon} | LM {record['language_loss']:.4f} | "
                f"token acc {100 * record['answer_token_accuracy']:6.2f}% | "
                f"entropy {record['router_entropy']:.3f} | {elapsed:.3f}s"
            )
        if step % checkpoint_every == 0 or step == horizon:
            payload = {
                "run_config": run_config, "step": step,
                "params": jax.device_get(params),
                "optimizer_state": jax.device_get(optimizer_state),
                "batcher_rng_state": batcher.rng.bit_generator.state,
                "history": history, "optimizer_seconds": optimizer_seconds,
                "compilation_seconds": compilation_seconds,
            }
            atomic_write_bytes(latest_path, pickle.dumps(payload, protocol=pickle.HIGHEST_PROTOCOL))
            atomic_write_json(history_path, history)
            print(f"saved checkpoint: {run_id} step {step}")

    validation = evaluate_moe_validation(params, evaluate_batch, generate_batch)
    examples_seen = horizon * BATCH_SIZE
    result = {
        "status": "complete", "run_config": run_config, "run_id": run_id,
        "architecture": "moe", "model_id": model_id,
        "dense_reference": config["dense_reference"], "horizon": horizon,
        "stored_parameters": config["stored_parameters"],
        "active_parameters": config["active_parameters"],
        "examples_seen": examples_seen,
        "input_tokens": examples_seen * MODEL_INPUT_LENGTH,
        "answer_tokens": examples_seen * ANSWER_DIGITS,
        "estimated_training_flops": (
            6 * config["active_parameters"] * examples_seen * MODEL_INPUT_LENGTH
        ),
        "final_training_loss": history[-1]["language_loss"],
        "final_training_token_accuracy": history[-1]["answer_token_accuracy"],
        "optimizer_seconds": optimizer_seconds,
        "compilation_seconds": compilation_seconds,
        **validation,
    }
    atomic_write_bytes(best_path, pickle.dumps({
        "run_config": run_config, "params": jax.device_get(params), "validation": validation
    }, protocol=pickle.HIGHEST_PROTOCOL))
    atomic_write_json(result_path, result)
    print(
        f"complete {run_id} | validation loss {validation['validation_loss']:.4f} | "
        f"exact {100 * validation['validation_exact_match']:.2f}%"
    )
    return result


## Run the 16 independent MoE experiments

Each coordinate runs in a fresh GPU process. Process exit releases all T4 memory before the next coordinate starts; completed results are checked and skipped, and incomplete checkpoints resume automatically. A one-step fresh-process `moe_active_10m_h0050` smoke test exercises the largest compilation first; the scientific batch size remains 2048.


In [ ]:
def worker_command(model_id, horizon, run_dir):
    return [
        sys.executable, "-u", str(WORKER),
        "--family", "moe",
        "--model-id", model_id,
        "--horizon", str(horizon),
        "--seed", str(SEED),
        "--output-dir", str(run_dir),
    ]

def launch_coordinate(model_id, config, horizon):
    run_id = f"{model_id}_h{horizon:04d}"
    run_dir = MOE_RUNS_ROOT / run_id
    result_path = run_dir / "result.json"
    expected = {"run_id": run_id, "model_id": model_id, "horizon": horizon,
                "seed": SEED, "batch_size": BATCH_SIZE, "model": config,
                "experts": N_EXPERTS, "top_k": TOP_K}

    if result_path.exists():
        result = json.loads(result_path.read_text())
        if result.get("status") == "complete" and result.get("run_config") == expected:
            print(f"skip {run_id}: complete")
            return result
        if result.get("status") == "complete":
            raise ValueError(f"existing result does not match this run: {run_id}")

    command = worker_command(model_id, horizon, run_dir)
    print("launch", shlex.join(command))
    completed = subprocess.run(command, cwd=REPO_DIR)
    if completed.returncode:
        failure_path = run_dir / "worker_failure.json"
        detail = failure_path.read_text() if failure_path.exists() else "no failure report"
        raise RuntimeError(f"worker failed for {run_id}:\n{detail}")

    result = json.loads(result_path.read_text())
    if result.get("status") != "complete" or result.get("run_config") != expected:
        raise ValueError(f"worker returned an incompatible result: {run_id}")
    return result

smoke_run_dir = MOE_RUNS_ROOT / "moe_active_10m_h0050"
smoke_result_path = smoke_run_dir / "result.json"
if not smoke_result_path.exists():
    print("Fresh-process moe_active_10m_h0050 smoke test")
    smoke_command = worker_command("moe_active_10m", 50, smoke_run_dir) + ["--smoke-test"]
    smoke = subprocess.run(smoke_command, cwd=REPO_DIR)
    if smoke.returncode:
        failure_path = smoke_run_dir / "worker_failure.json"
        detail = failure_path.read_text() if failure_path.exists() else "no failure report"
        raise RuntimeError(f"largest-model smoke test failed:\n{detail}")

moe_results = []
for model_id, config in MOE_CONFIGS.items():
    for horizon in HORIZONS:
        moe_results.append(launch_coordinate(model_id, config, horizon))
        atomic_write_json(MOE_RESULTS_PATH, {
            "status": "complete" if len(moe_results) == 16 else "in_progress",
            "results": moe_results,
        })

assert len(moe_results) == 16
assert all(result["status"] == "complete" for result in moe_results)
atomic_write_json(MOE_RESULTS_PATH, {"status": "complete", "results": moe_results})
print("MoE study complete:", MOE_RESULTS_PATH)


## MoE result tables and routing diagnostics


In [ ]:
moe_frame = pd.DataFrame(moe_results).sort_values(["active_parameters", "horizon"])
quality_table = moe_frame[[
    "model_id", "horizon", "validation_loss", "validation_token_accuracy",
    "validation_exact_match", "router_entropy", "balance_loss", "z_loss",
]].copy()
quality_table["validation_token_accuracy"] *= 100
quality_table["validation_exact_match"] *= 100
timing_table = moe_frame[[
    "model_id", "horizon", "stored_parameters", "active_parameters", "input_tokens",
    "estimated_training_flops", "optimizer_seconds", "validation_seconds",
]].copy()
timing_table["examples_per_second"] = moe_frame["examples_seen"] / moe_frame["optimizer_seconds"]

display(quality_table.style.format({
    "validation_loss": "{:.5f}", "validation_token_accuracy": "{:.2f}%",
    "validation_exact_match": "{:.2f}%", "router_entropy": "{:.3f}",
    "balance_loss": "{:.3f}", "z_loss": "{:.3f}",
}))
display(timing_table.style.format({
    "stored_parameters": "{:,}", "active_parameters": "{:,}", "input_tokens": "{:,}",
    "estimated_training_flops": "{:.3e}", "optimizer_seconds": "{:.1f}",
    "validation_seconds": "{:.1f}", "examples_per_second": "{:,.0f}",
}))

route_rows = []
for result in moe_results:
    layer_routes = np.asarray(result["route_fraction"])
    for layer in range(layer_routes.shape[0]):
        route_rows.append({"model_id": result["model_id"], "horizon": result["horizon"],
                           "layer": layer, **{f"expert_{expert}": layer_routes[layer, expert]
                           for expert in range(N_EXPERTS)}})
route_frame = pd.DataFrame(route_rows)
display(route_frame.head(12).style.format({f"expert_{expert}": "{:.3f}"
                                           for expert in range(N_EXPERTS)}))

figure, axes = plt.subplots(1, 2, figsize=(12, 4.2))
mean_routes = route_frame.groupby("model_id")[[f"expert_{e}" for e in range(N_EXPERTS)]].mean()
mean_routes.plot(kind="bar", ax=axes[0])
axes[0].axhline(1 / N_EXPERTS, linestyle="--", color="black", linewidth=1)
axes[0].set(ylabel="top-2 route fraction", title="Mean expert traffic")
for model_id, group in moe_frame.groupby("model_id", sort=False):
    axes[1].plot(group["horizon"], group["router_entropy"], marker="o", label=model_id)
axes[1].axhline(math.log(N_EXPERTS), linestyle="--", color="black", linewidth=1)
axes[1].set(xlabel="independent horizon", ylabel="entropy", title="Router entropy")
axes[1].legend(fontsize=8)
figure.tight_layout()
save_figure(figure, "moe_routing_diagnostics")


## Dense-versus-MoE result tables


In [ ]:
assert DENSE_RESULTS_PATH.exists(), (
    "Notebook 02 results are missing. Run notebook 02 to completion before this comparison."
)
dense_payload = json.loads(DENSE_RESULTS_PATH.read_text())
assert dense_payload.get("status") == "complete"
assert len(dense_payload.get("results", [])) == 16
assert len(moe_results) == 16

dense_frame = pd.DataFrame(dense_payload["results"])
pairs = dense_frame.merge(
    moe_frame,
    left_on=["model_id", "horizon"],
    right_on=["dense_reference", "horizon"],
    suffixes=("_dense", "_moe"),
    validate="one_to_one",
)
assert len(pairs) == 16
pairs["moe / dense loss"] = pairs["validation_loss_moe"] / pairs["validation_loss_dense"]
pairs["MoE - dense exact match"] = (
    pairs["validation_exact_match_moe"] - pairs["validation_exact_match_dense"]
)
pairs["moe / dense optimizer time"] = (
    pairs["optimizer_seconds_moe"] / pairs["optimizer_seconds_dense"]
)
pairs["active parameter ratio"] = pairs["active_parameters"] / pairs["parameter_count"]

comparison_table = pairs[[
    "model_id_dense", "model_id_moe", "horizon", "validation_loss_dense",
    "validation_loss_moe", "moe / dense loss", "validation_exact_match_dense",
    "validation_exact_match_moe", "MoE - dense exact match",
]].copy()
comparison_table[["validation_exact_match_dense", "validation_exact_match_moe",
                  "MoE - dense exact match"]] *= 100
compute_timing_table = pairs[[
    "model_id_dense", "horizon", "parameter_count", "stored_parameters",
    "active_parameters", "estimated_training_flops_dense",
    "estimated_training_flops_moe", "optimizer_seconds_dense",
    "optimizer_seconds_moe", "moe / dense optimizer time",
]].copy()

display(comparison_table.style.format({
    "validation_loss_dense": "{:.5f}", "validation_loss_moe": "{:.5f}",
    "moe / dense loss": "{:.3f}", "validation_exact_match_dense": "{:.2f}%",
    "validation_exact_match_moe": "{:.2f}%", "MoE - dense exact match": "{:+.2f} pp",
}))
display(compute_timing_table.style.format({
    "parameter_count": "{:,}", "stored_parameters": "{:,}", "active_parameters": "{:,}",
    "estimated_training_flops_dense": "{:.3e}", "estimated_training_flops_moe": "{:.3e}",
    "optimizer_seconds_dense": "{:.1f}", "optimizer_seconds_moe": "{:.1f}",
    "moe / dense optimizer time": "{:.3f}",
}))
atomic_write_json(COMPARISON_ROOT / "paired_results.json", pairs.to_dict(orient="records"))


## Dense-versus-MoE figures


In [ ]:
combined = pd.concat([
    dense_frame.assign(plot_architecture="dense", plot_model=dense_frame["model_id"],
                       plot_compute=dense_frame["estimated_training_flops"]),
    moe_frame.assign(plot_architecture="moe", plot_model=moe_frame["model_id"],
                     plot_compute=moe_frame["estimated_training_flops"]),
], ignore_index=True)

def empirical_frontier(frame):
    ordered = frame.sort_values(["plot_compute", "validation_loss"])
    keep, best = [], math.inf
    for index, row in ordered.iterrows():
        if row["validation_loss"] < best:
            keep.append(index)
            best = row["validation_loss"]
    return ordered.loc[keep]

frontiers = {
    architecture: empirical_frontier(combined[combined["plot_architecture"] == architecture])
    for architecture in ("dense", "moe")
}
figure, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for architecture, frontier in frontiers.items():
    axes[0].plot(frontier["plot_compute"], frontier["validation_loss"],
                 marker="o", label=architecture)
for architecture, group in combined.groupby("plot_architecture"):
    axes[1].scatter(group["input_tokens"], 100 * group["validation_exact_match"],
                    label=architecture, alpha=0.8)
axes[2].scatter(pairs["optimizer_seconds_dense"], pairs["optimizer_seconds_moe"],
                c=np.log10(pairs["parameter_count"]), cmap="viridis")
timing_limit = max(pairs["optimizer_seconds_dense"].max(), pairs["optimizer_seconds_moe"].max())
axes[2].plot([0, timing_limit], [0, timing_limit], linestyle="--", color="black", linewidth=1)
axes[0].set(xscale="log", yscale="log", xlabel="estimated training FLOPs",
            ylabel="validation loss", title="Empirical frontiers")
axes[0].legend()
axes[1].set(xscale="log", xlabel="input-token exposure", ylabel="exact match (%)",
            title="Validation exact match")
axes[1].legend()
axes[2].set(xlabel="dense optimizer seconds", ylabel="MoE optimizer seconds",
            title="Measured training time")
figure.tight_layout()
save_figure(figure, "dense_moe_comparison")


## Conclusions from completed results


In [ ]:
assert len(pairs) == 16
assert pairs[["validation_loss_dense", "validation_loss_moe"]].notna().all().all()
moe_loss_wins = int((pairs["validation_loss_moe"] < pairs["validation_loss_dense"]).sum())
moe_exact_wins = int(
    (pairs["validation_exact_match_moe"] > pairs["validation_exact_match_dense"]).sum()
)
median_time_ratio = float(pairs["moe / dense optimizer time"].median())
best_dense = dense_frame.loc[dense_frame["validation_loss"].idxmin()]
best_moe = moe_frame.loc[moe_frame["validation_loss"].idxmin()]

print("Conclusions from completed dense and MoE runs")
print(f"MoE has lower validation loss at {moe_loss_wins} of 16 matched coordinates.")
print(f"MoE has higher exact match at {moe_exact_wins} of 16 matched coordinates.")
print(f"Median MoE/dense optimizer-time ratio: {median_time_ratio:.3f}×.")
print(
    f"Best dense loss: {best_dense['validation_loss']:.5f} "
    f"({best_dense['model_id']}, {int(best_dense['horizon'])} steps)."
)
print(
    f"Best MoE loss: {best_moe['validation_loss']:.5f} "
    f"({best_moe['model_id']}, {int(best_moe['horizon'])} steps)."
)
